# QUBO Step-by-Step Formulation

This notebook is intentionally independent from `src/quantum/qubo_builder.py`.

The goal is pedagogical: build the QUBO coefficient dictionary term by term so the formulation can be explained in a meeting.

We build:

$$
H_{\text{total}} = H_{\text{energy}} + H_{\text{assign}} + H_{\text{GPU}} + H_{\text{peak}}
$$

The notebook supports two PUE modes:

- `fixed`: fixed or exogenous PUE multiplies IT load.
- `load_dependent`: affine utilization-dependent PUE is used in the energy term.

The load-dependent PUE mode is kept out of the squared peak term because that would create quartic terms, not a plain QUBO.

## 1. Toy Instance and Scenario Knobs

We use a tiny instance so every coefficient can be inspected.

The key switch is `PUE_MODE`:

```python
PUE_MODE = "fixed"
# or
PUE_MODE = "load_dependent"
```

For load-dependent PUE, the energy term uses:

$$
\mathrm{PUE}_t(x) = PUE_{\text{high}} - (PUE_{\text{high}} - PUE_{\text{low}}) \frac{L^{\mathrm{IT}}_t(x)}{L^{\mathrm{IT}}_{\max}}
$$

In [34]:
from __future__ import annotations

from dataclasses import dataclass, field
from itertools import combinations, product
from math import ceil, log2

import pandas as pd

jobs = pd.DataFrame([
    {"job_id": "j1", "duration": 1, "earliest_start": 0, "latest_start": 1, "gpu": 1, "power_mw": 0.010},
    {"job_id": "j2", "duration": 1, "earliest_start": 0, "latest_start": 2, "gpu": 1, "power_mw": 0.015},
])

clusters = pd.DataFrame([
    {"cluster_id": "c1", "gpu_capacity": 2},
])

hours = [0, 1, 2]
delta_t = 1.0

# Effective price already represents the simplified QUBO view of energy timing.
price = {0: 30.0, 1: 100.0, 2: 40.0}

# PUE mode: "fixed" or "load_dependent".
PUE_MODE = "fixed"

# Fixed/exogenous PUE profile.
pue = {0: 1.20, 1: 1.20, 2: 1.20}

# Load-dependent PUE calibration.
pue_high = 1.35
pue_low = 1.15
it_load_max = jobs["power_mw"].sum()

weights = {
    "assignment": 200.0,
    "gpu_capacity": 80.0,
    "peak": 1.0,
}

jobs

,job_id,duration,earliest_start,latest_start,gpu,power_mw
0,j1,1,0,1,1,0.010
1,j2,1,0,2,1,0.015


## 2. Basic QUBO Data Structure

A QUBO has the form:

$$
H(z) = q_0 + \sum_i q_i z_i + \sum_{i < j} q_{ij} z_i z_j
$$

The `QuboState` object stores the coefficients and variable metadata. Every function receives this state explicitly and returns or mutates it explicitly; no function depends on hidden global QUBO variables.

In [35]:
@dataclass
class QuboState:
    """Small didactic QUBO container."""

    linear: dict[int, float] = field(default_factory=dict)
    quadratic: dict[tuple[int, int], float] = field(default_factory=dict)
    offset: float = 0.0
    variables: list[dict] = field(default_factory=list)


def add_linear(state: QuboState, i: int, coeff: float) -> None:
    """Add q_i z_i to the QUBO."""
    if abs(coeff) > 1e-12:
        state.linear[i] = state.linear.get(i, 0.0) + float(coeff)


def add_quadratic(state: QuboState, i: int, j: int, coeff: float) -> None:
    """Add q_ij z_i z_j to the QUBO."""
    if abs(coeff) <= 1e-12:
        return
    if i == j:
        # Binary identity: z_i^2 = z_i.
        add_linear(state, i, coeff)
        return
    key = (i, j) if i < j else (j, i)
    state.quadratic[key] = state.quadratic.get(key, 0.0) + float(coeff)


def add_squared_penalty(state: QuboState, terms: dict[int, float], rhs: float, weight: float) -> None:
    """Add weight * (sum_i terms[i] z_i - rhs)^2."""
    state.offset += weight * rhs * rhs
    for i, ai in terms.items():
        add_linear(state, i, weight * (ai * ai - 2.0 * rhs * ai))
    for i, j in combinations(terms, 2):
        add_quadratic(state, i, j, 2.0 * weight * terms[i] * terms[j])


def qubo_summary(state: QuboState) -> dict:
    """Return a compact summary of the current QUBO."""
    return {
        "variables": len(state.variables),
        "linear_terms": len(state.linear),
        "quadratic_terms": len(state.quadratic),
        "offset": state.offset,
    }

state = QuboState()
qubo_summary(state)

{'variables': 0, 'linear_terms': 0, 'quadratic_terms': 0, 'offset': 0.0}

## 3. Assignment Variables

Create one variable per feasible scheduling option:

$$
x_{i,k,s} = 1
$$

if job $i$ starts on cluster $k$ at start slot $s$.

In [36]:
def feasible_starts(job: pd.Series) -> list[int]:
    """Return feasible start slots for one job."""
    return list(range(int(job.earliest_start), int(job.latest_start) + 1))


def add_assignment_variables(state: QuboState, jobs_df: pd.DataFrame, clusters_df: pd.DataFrame) -> None:
    """Create x_{i,k,s} variables for all feasible toy-instance options."""
    for job in jobs_df.itertuples(index=False):
        for cluster in clusters_df.itertuples(index=False):
            for start in range(int(job.earliest_start), int(job.latest_start) + 1):
                state.variables.append({
                    "type": "assignment",
                    "job_id": job.job_id,
                    "cluster_id": cluster.cluster_id,
                    "start": int(start),
                    "duration": int(job.duration),
                    "gpu": float(job.gpu),
                    "power_mw": float(job.power_mw),
                })

add_assignment_variables(state, jobs, clusters)
assignment_variable_count = len(state.variables)
pd.DataFrame(state.variables)

,type,job_id,cluster_id,start,duration,gpu,power_mw
0,assignment,j1,c1,0,1,1.0,0.010
1,assignment,j1,c1,1,1,1.0,0.010
2,assignment,j2,c1,0,1,1.0,0.015
3,assignment,j2,c1,1,1,1.0,0.015
4,assignment,j2,c1,2,1,1.0,0.015


## 4. Activity Coefficient

The activity coefficient is precomputed data, not a decision variable:

$$
a_{i,s,t} =
\begin{cases}
1, & s \le t < s + d_i \\
0, & \text{otherwise}
\end{cases}
$$

In [37]:
def is_active(start: int, duration: int, hour: int) -> bool:
    """Return whether a job starting at `start` is active at `hour`."""
    return start <= hour < start + duration


def activity_table(state: QuboState, hours: list[int]) -> pd.DataFrame:
    """Build a readable activity table for assignment variables."""
    rows = []
    for idx, var in enumerate(state.variables):
        if var["type"] != "assignment":
            continue
        for t in hours:
            rows.append({
                "var": idx,
                "job": var["job_id"],
                "start": var["start"],
                "hour": t,
                "active": int(is_active(var["start"], var["duration"], t)),
            })
    return pd.DataFrame(rows)

activity_table(state, hours).head(20)

,var,job,start,hour,active
0,0,j1,0,0,1
1,0,j1,0,1,0
2,0,j1,0,2,0
3,1,j1,1,0,0
4,1,j1,1,1,1
5,1,j1,1,2,0
6,2,j2,0,0,1
7,2,j2,0,1,0
8,2,j2,0,2,0
9,3,j2,1,0,0


## 5. Energy Cost Term

The function below chooses the PUE mode.

Fixed PUE:

$$
H_{\text{energy}}^{\text{PUE-fixed}} = \sum_{i,k,s} \left( \sum_t c_t \Delta t \, \mathrm{PUE}_t \, p_i \, a_{i,s,t} \right) x_{i,k,s}
$$

Load-dependent PUE energy:

$$
H_{\text{facility}} = \sum_t c_t \Delta t \left[ PUE_{\text{high}} L^{\mathrm{IT}}_t(x) - \frac{PUE_{\text{high}} - PUE_{\text{low}}}{L^{\mathrm{IT}}_{\max}} \left(L^{\mathrm{IT}}_t(x)\right)^2 \right]
$$

The load-dependent case is still QUBO-compatible for the energy term because it produces only linear and quadratic coefficients.

In [38]:
def active_assignment_terms(state: QuboState, hours: list[int], hour: int, coefficient_field: str) -> dict[int, float]:
    """Return coefficients for assignment variables active in one hour."""
    terms = {}
    for idx, var in enumerate(state.variables):
        if var["type"] != "assignment":
            continue
        if is_active(var["start"], var["duration"], hour):
            terms[idx] = float(var[coefficient_field])
    return terms


def add_energy_cost(
    state: QuboState,
    hours: list[int],
    price: dict[int, float],
    delta_t: float,
    pue_mode: str,
    fixed_pue: dict[int, float] | None = None,
    pue_high: float | None = None,
    pue_low: float | None = None,
    it_load_max: float | None = None,
) -> None:
    """Add either fixed-PUE or load-dependent-PUE energy cost."""
    if pue_mode == "fixed":
        if fixed_pue is None:
            raise ValueError("fixed_pue is required when pue_mode='fixed'")
        for t in hours:
            active_power = active_assignment_terms(state, hours, t, "power_mw")
            for idx, power_mw in active_power.items():
                add_linear(state, idx, price[t] * delta_t * fixed_pue[t] * power_mw)
        return

    if pue_mode == "load_dependent":
        if pue_high is None or pue_low is None or it_load_max is None:
            raise ValueError("pue_high, pue_low, and it_load_max are required for load-dependent PUE")
        beta = pue_high - pue_low
        for t in hours:
            active_power = active_assignment_terms(state, hours, t, "power_mw")

            # Linear part: c_t dt PUE_high L_t.
            for idx, power_mw in active_power.items():
                add_linear(state, idx, price[t] * delta_t * pue_high * power_mw)

            # Quadratic part: - c_t dt beta / Lmax * L_t^2.
            factor = -price[t] * delta_t * beta / it_load_max
            for idx, power_mw in active_power.items():
                add_linear(state, idx, factor * power_mw * power_mw)
            for i, j in combinations(active_power, 2):
                add_quadratic(state, i, j, 2.0 * factor * active_power[i] * active_power[j])
        return

    raise ValueError("pue_mode must be 'fixed' or 'load_dependent'")

add_energy_cost(
    state,
    hours=hours,
    price=price,
    delta_t=delta_t,
    pue_mode=PUE_MODE,
    fixed_pue=pue,
    pue_high=pue_high,
    pue_low=pue_low,
    it_load_max=it_load_max,
)

qubo_summary(state)

{'variables': 5, 'linear_terms': 5, 'quadratic_terms': 0, 'offset': 0.0}

In [39]:
pd.DataFrame([{"var": i, "linear_coeff": coeff, **state.variables[i]} for i, coeff in sorted(state.linear.items())])

,var,linear_coeff,type,job_id,cluster_id,start,duration,gpu,power_mw
0,0,0.36,assignment,j1,c1,0,1,1.0,0.010
1,1,1.20,assignment,j1,c1,1,1,1.0,0.010
2,2,0.54,assignment,j2,c1,0,1,1.0,0.015
3,3,1.80,assignment,j2,c1,1,1,1.0,0.015
4,4,0.72,assignment,j2,c1,2,1,1.0,0.015


## 6. Assignment Constraint

Each job must be assigned exactly once:

$$
\sum_{(k,s) \in \mathcal{A}_i} x_{i,k,s} = 1
$$

QUBO penalty:

$$
H_{\text{assign}} = \lambda_A \sum_i \left( \sum_{(k,s) \in \mathcal{A}_i} x_{i,k,s} - 1 \right)^2
$$

In [40]:
def add_assignment_penalty(state: QuboState, jobs_df: pd.DataFrame, weight: float) -> None:
    """Add one-hot assignment penalties, one job at a time."""
    for job_id in jobs_df["job_id"]:
        terms = {
            idx: 1.0
            for idx, var in enumerate(state.variables)
            if var["type"] == "assignment" and var["job_id"] == job_id
        }
        add_squared_penalty(state, terms, rhs=1.0, weight=weight)

add_assignment_penalty(state, jobs, weights["assignment"])
qubo_summary(state)

{'variables': 5, 'linear_terms': 5, 'quadratic_terms': 4, 'offset': 400.0}

## 7. GPU Capacity Constraint With Binary Slack

GPU demand per cluster and time:

$$
D^G_{k,t}(x) = \sum_i \sum_s g_i a_{i,s,t} x_{i,k,s}
$$

Capacity with slack:

$$
D^G_{k,t}(x) + \sum_{b=0}^{B_k-1} 2^b y_{k,t,b} = G_k
$$

Penalty:

$$
H_{\text{GPU}} = \lambda_G \sum_{k,t} \left( D^G_{k,t}(x) + \sum_b 2^b y_{k,t,b} - G_k \right)^2
$$

In [41]:
def add_gpu_capacity_penalty_with_slack(
    state: QuboState,
    clusters_df: pd.DataFrame,
    hours: list[int],
    weight: float,
) -> None:
    """Add GPU capacity penalties using binary unused-GPU slack bits."""
    for cluster in clusters_df.itertuples(index=False):
        G = int(cluster.gpu_capacity)
        bits = max(1, ceil(log2(G + 1)))
        for t in hours:
            terms = {}
            for idx, var in enumerate(state.variables):
                if var["type"] != "assignment":
                    continue
                if var["cluster_id"] != cluster.cluster_id:
                    continue
                if is_active(var["start"], var["duration"], t):
                    terms[idx] = float(var["gpu"])

            if not terms:
                continue

            for b in range(bits):
                slack_idx = len(state.variables)
                state.variables.append({
                    "type": "gpu_slack",
                    "cluster_id": cluster.cluster_id,
                    "hour": t,
                    "bit": b,
                    "coefficient": float(2**b),
                })
                terms[slack_idx] = float(2**b)

            add_squared_penalty(state, terms, rhs=float(G), weight=weight)

variables_before_slack = len(state.variables)
add_gpu_capacity_penalty_with_slack(state, clusters, hours, weights["gpu_capacity"])
slack_variable_count = len(state.variables) - variables_before_slack

{"slack_variables_added": slack_variable_count, **qubo_summary(state)}

{'slack_variables_added': 6,
 'variables': 11,
 'linear_terms': 11,
 'quadratic_terms': 19,
 'offset': 1360.0}

In [42]:
pd.DataFrame(state.variables).query("type == 'gpu_slack'")

,type,job_id,cluster_id,start,duration,gpu,power_mw,hour,bit,coefficient
5,gpu_slack,NaN,c1,NaN,NaN,NaN,NaN,0.0,0.0,1.0
6,gpu_slack,NaN,c1,NaN,NaN,NaN,NaN,0.0,1.0,2.0
7,gpu_slack,NaN,c1,NaN,NaN,NaN,NaN,1.0,0.0,1.0
8,gpu_slack,NaN,c1,NaN,NaN,NaN,NaN,1.0,1.0,2.0
9,gpu_slack,NaN,c1,NaN,NaN,NaN,NaN,2.0,0.0,1.0
10,gpu_slack,NaN,c1,NaN,NaN,NaN,NaN,2.0,1.0,2.0


## 8. Peak Penalty

For fixed PUE, we use squared facility load:

$$
H_{\text{peak}}^{\text{PUE-fixed}} = \lambda_P \sum_t \left( \mathrm{PUE}_t L^{\mathrm{IT}}_t(x) \right)^2
$$

For load-dependent PUE, we intentionally keep the peak proxy on IT load:

$$
H_{\text{peak}}^{\mathrm{IT}} = \lambda_P \sum_t \left( L^{\mathrm{IT}}_t(x) \right)^2
$$

Reason: load-dependent facility power already contains $\left(L^{\mathrm{IT}}_t(x)\right)^2$. Squaring it would create quartic terms.

In [43]:
def add_peak_penalty(
    state: QuboState,
    hours: list[int],
    weight: float,
    pue_mode: str,
    fixed_pue: dict[int, float] | None = None,
) -> None:
    """Add peak proxy according to the selected PUE mode."""
    for t in hours:
        active_power = active_assignment_terms(state, hours, t, "power_mw")
        if not active_power:
            continue

        if pue_mode == "fixed":
            if fixed_pue is None:
                raise ValueError("fixed_pue is required when pue_mode='fixed'")
            terms = {idx: fixed_pue[t] * power_mw for idx, power_mw in active_power.items()}
        elif pue_mode == "load_dependent":
            terms = active_power
        else:
            raise ValueError("pue_mode must be 'fixed' or 'load_dependent'")

        add_squared_penalty(state, terms, rhs=0.0, weight=weight)

add_peak_penalty(state, hours, weights["peak"], PUE_MODE, fixed_pue=pue)
qubo_summary(state)

{'variables': 11, 'linear_terms': 11, 'quadratic_terms': 19, 'offset': 1360.0}

## 9. Inspect Final QUBO Coefficients

At this point the QUBO has all terms needed for the selected PUE mode.

In [44]:
linear_df = pd.DataFrame([{"var": i, "coefficient": coeff, **state.variables[i]} for i, coeff in sorted(state.linear.items())])
quadratic_df = pd.DataFrame([
    {"left": i, "right": j, "coefficient": coeff}
    for (i, j), coeff in sorted(state.quadratic.items())
])

qubo_summary(state)

{'variables': 11, 'linear_terms': 11, 'quadratic_terms': 19, 'offset': 1360.0}

In [45]:
linear_df.head(20)

,var,coefficient,type,job_id,cluster_id,start,duration,gpu,power_mw,hour,bit
0,0,-439.639856,assignment,j1,c1,0.0,1.0,1.0,0.010,NaN,NaN
1,1,-438.799856,assignment,j1,c1,1.0,1.0,1.0,0.010,NaN,NaN
2,2,-439.459676,assignment,j2,c1,0.0,1.0,1.0,0.015,NaN,NaN
3,3,-438.199676,assignment,j2,c1,1.0,1.0,1.0,0.015,NaN,NaN
4,4,-439.279676,assignment,j2,c1,2.0,1.0,1.0,0.015,NaN,NaN
5,5,1.000000,gpu_slack,NaN,c1,NaN,NaN,NaN,NaN,0.0,0.0
6,6,2.000000,gpu_slack,NaN,c1,NaN,NaN,NaN,NaN,0.0,1.0
7,7,1.000000,gpu_slack,NaN,c1,NaN,NaN,NaN,NaN,1.0,0.0
8,8,2.000000,gpu_slack,NaN,c1,NaN,NaN,NaN,NaN,1.0,1.0
9,9,1.000000,gpu_slack,NaN,c1,NaN,NaN,NaN,NaN,2.0,0.0


In [46]:
quadratic_df.head(20)

,left,right,coefficient
0,0,1,400.000000
1,0,2,160.000432
2,0,5,160.000000
3,0,6,320.000000
4,1,3,160.000432
5,1,7,160.000000
6,1,8,320.000000
7,2,3,400.000000
8,2,4,400.000000
9,2,5,160.000000


## 10. Evaluate and Decode a Candidate Sample

A QUBO solver returns binary values for all assignment and slack variables. We decode only assignment variables into a schedule.

In [47]:
def qubo_energy(state: QuboState, sample: dict[int, int]) -> float:
    """Evaluate the QUBO for a binary sample."""
    value = state.offset
    value += sum(coeff * sample.get(i, 0) for i, coeff in state.linear.items())
    value += sum(coeff * sample.get(i, 0) * sample.get(j, 0) for (i, j), coeff in state.quadratic.items())
    return float(value)


def decode_schedule(state: QuboState, sample: dict[int, int]) -> pd.DataFrame:
    """Decode selected assignment variables into a schedule."""
    rows = []
    for idx, bit in sample.items():
        if bit != 1:
            continue
        var = state.variables[idx]
        if var["type"] != "assignment":
            continue
        rows.append({"var": idx, **var})
    return pd.DataFrame(rows)

sample = {i: 0 for i in range(len(state.variables))}
for idx, var in enumerate(state.variables):
    if var["type"] == "assignment" and (var["job_id"], var["start"]) in {("j1", 0), ("j2", 2)}:
        sample[idx] = 1

# Choose a simple slack assignment. The simulator below will search all bits exactly.
for idx, var in enumerate(state.variables):
    if var["type"] == "gpu_slack" and var["bit"] == 0:
        sample[idx] = 1

{"energy": qubo_energy(state, sample), "selected_variables": [i for i, bit in sample.items() if bit == 1]}

{'energy': 81.08046800000011, 'selected_variables': [0, 4, 5, 7, 9]}

In [48]:
decode_schedule(state, sample)

,var,type,job_id,cluster_id,start,duration,gpu,power_mw
0,0,assignment,j1,c1,0,1,1.0,0.010
1,4,assignment,j2,c1,2,1,1.0,0.015


## 11. Exact Brute-Force Simulator

For this tiny example, we can simulate the QUBO by brute force: enumerate all binary strings and keep the minimum-energy solution.

This is not scalable, but it is excellent for explaining and validating the QUBO logic.

In [49]:
def solve_qubo_bruteforce(state: QuboState, max_variables: int = 24) -> dict:
    """Solve a tiny QUBO exactly by enumerating every binary string."""
    n = len(state.variables)
    if n > max_variables:
        raise ValueError(f"Brute force would require 2^{n} samples; reduce the toy instance.")

    best_energy = float("inf")
    best_sample = None
    for bits in product([0, 1], repeat=n):
        candidate = dict(enumerate(bits))
        energy = qubo_energy(state, candidate)
        if energy < best_energy:
            best_energy = energy
            best_sample = candidate

    return {"energy": best_energy, "sample": best_sample, "schedule": decode_schedule(state, best_sample)}

bruteforce_result = solve_qubo_bruteforce(state)
{
    "energy": bruteforce_result["energy"],
    "selected_variables": [i for i, bit in bruteforce_result["sample"].items() if bit == 1],
}

{'energy': 0.9008999999998082, 'selected_variables': [0, 2, 8, 10]}

In [50]:
bruteforce_result["schedule"]

,var,type,job_id,cluster_id,start,duration,gpu,power_mw
0,0,assignment,j1,c1,0,1,1.0,0.010
1,2,assignment,j2,c1,0,1,1.0,0.015


## 12. Optional Simulated Annealing With D-Wave Ocean

If `dimod` and `neal` are installed, this cell runs a classical simulated annealer using the same QUBO. If they are not installed, the exact brute-force simulator above is the fallback.

In [51]:
def to_dimod_bqm(state: QuboState):
    """Convert the didactic QUBO into a dimod BinaryQuadraticModel."""
    import dimod

    bqm = dimod.BinaryQuadraticModel({}, {}, state.offset, dimod.BINARY)
    for i, coeff in state.linear.items():
        bqm.add_variable(i, coeff)
    for (i, j), coeff in state.quadratic.items():
        bqm.add_interaction(i, j, coeff)
    return bqm


def solve_with_neal(state: QuboState, reads: int = 200) -> dict:
    """Run D-Wave Ocean simulated annealing if optional dependencies exist."""
    import neal

    bqm = to_dimod_bqm(state)
    sampleset = neal.SimulatedAnnealingSampler().sample(bqm, num_reads=reads)
    best = sampleset.first
    sample = {int(i): int(v) for i, v in best.sample.items()}
    return {"energy": float(best.energy), "sample": sample, "schedule": decode_schedule(state, sample)}

try:
    neal_result = solve_with_neal(state)
except ImportError as exc:
    neal_result = {"status": "optional_dependency_missing", "message": str(exc)}

neal_result if "status" in neal_result else {"energy": neal_result["energy"]}

{'energy': 0.9009000000000924}

In [52]:
if isinstance(neal_result, dict) and "schedule" in neal_result:
    try:
        display(neal_result["schedule"])
    except NameError:
        print(neal_result["schedule"])


,var,type,job_id,cluster_id,start,duration,gpu,power_mw
0,0,assignment,j1,c1,0,1,1.0,0.010
1,2,assignment,j2,c1,0,1,1.0,0.015
